# 🧠 Vigyan-2B: GRPO Reinforcement Learning Reasoning Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyansh001boy-tech/vigyan-7b-demo/blob/main/vigyan_2b_grpo_reasoning_demo.ipynb)
&nbsp;[![License: CC BY-NC 4.0](https://img.shields.io/badge/License-CC_BY--NC_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc/4.0/)
&nbsp;[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-vigyan-2b-reasoning-grpo-yellow)](https://huggingface.co/shreyansh12183/vigyan-2b-reasoning-grpo)

---

### 🧠 Architectural Overview
**vigyan-2b-reasoning-grpo** evaluates **Group Relative Policy Optimization (GRPO)** — the policy gradient reinforcement learning algorithm utilized by DeepSeekMath and DeepSeek-R1.

* **Objective:** Elicits autonomous test-time compute scratchpad reasoning tokens (`<think> ... </think>`) without teacher distillation.
* **Reinforcement Learning:** Aligned with policy gradient reward models targeting exact numerical derivations.

> **License:** Creative Commons Attribution-NonCommercial 4.0 International (**CC BY-NC 4.0**). Free for non-commercial academic research and student study.

### Step 1: Install Dependencies (Runs in ~30s on free Colab T4)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes gradio
print('✅ Environment ready.')

### Step 2: Load Model in 4-bit NF4 Precision (Consumes < 3 GB VRAM)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'shreyansh12183/Shreyansh-STEM-AI-2B-v3'
ADAPTER_ID = 'shreyansh12183/vigyan-2b-reasoning-grpo'

print(f'Loading Base Model {BASE_MODEL} + GRPO Policy Adapter {ADAPTER_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto'
)
model = PeftModel.from_pretrained(base, ADAPTER_ID)
print('✅ 2B GRPO Reasoning Model loaded in ~2.1 GB VRAM!')


### Step 3: Launch Interactive Gradio WebUI

In [ ]:
import gradio as gr

def generate_response(prompt, temperature, max_tokens):
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 1e-2),
            do_sample=temperature > 0.05,
            pad_token_id=tokenizer.eos_token_id
        )
    gen = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return gen

demo = gr.Interface(
    fn=generate_response,
    inputs=[
        gr.Textbox(lines=4, label='Scientific / STEM Query', placeholder='Enter your problem...'),
        gr.Slider(0.0, 1.0, value=0.2, step=0.05, label='Temperature'),
        gr.Slider(64, 1024, value=512, step=64, label='Max Tokens')
    ],
    outputs=gr.Textbox(lines=10, label='Model Output & Mathematical Derivation'),
    title='🧠 Vigyan-2B: GRPO Reinforcement Learning Reasoning Model',
    description='Non-commercial research demonstration powered by Vigyan AI Sovereign Infrastructure under CC BY-NC 4.0.',
    examples=[['Janet buys a dress with a 20% discount. If she paid $40, what was the original price?'], ['What is the remainder when 3^100 is divided by 7? Show step-by-step modular arithmetic.'], ['Find the value of k such that the quadratic equation 2x^2 + kx + 3 = 0 has two equal real roots.']]
)

demo.launch(share=True, debug=False)
